# 13 — LLMs abertas (Qwen2.5-7B + Llama-3.1-8B) no test set

Substitui o backend Sabia 4 (API) por dois LLMs abertos rodando localmente em Colab L4 (24 GB VRAM) em FP16, **sem quantizacao**:

- `Qwen/Qwen2.5-7B-Instruct` (~7.6B params, ~15 GB FP16) — Apache 2.0, multilingual, recomendado pela orientadora.
- `meta-llama/Llama-3.1-8B-Instruct` (~8B params, ~16 GB FP16) — Llama 3 community license (gated, aceitar termos no HF), baseline de referencia.

Ambos com o **mesmo prompt zero-shot** definido em `economy_classifier.llm_review.SYSTEM_PROMPT`. Resultados emitidos no formato padrao (`predictions.csv` + `result_card.json`) para entrar na tabela comparativa final.

## 0. Verificacao de GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU nao detectada. Va em Runtime > Change runtime type > GPU (L4 ou A100).",
    )

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
print(f"GPU: {GPU_NAME} ({VRAM_GB} GB VRAM)")
print(f"CUDA: {torch.version.cuda}")

HARDWARE = f"Colab-{GPU_NAME.split()[-1]}"

# L4 (24 GB) e suficiente para 7-8B em FP16. Em GPUs menores troque para 4-bit
# (USE_4BIT=True abaixo) ou use modelos menores.
if VRAM_GB < 20:
    print(f"\nAVISO: {GPU_NAME} tem pouca VRAM ({VRAM_GB} GB). "
          f"Considere LOAD_DTYPE='int8' ou USE_4BIT=True.")


## 1. Bootstrap (Colab + Local)

In [ ]:
import subprocess
import sys
import zipfile
from pathlib import Path


def _run(cmd: list[str], description: str) -> None:
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr, file=sys.stderr)
        raise RuntimeError(f"{description} failed with exit code {result.returncode}")


IN_COLAB = "google.colab" in sys.modules
print("Ambiente:", "Google Colab" if IN_COLAB else "Local")
print("Python   :", sys.version.split()[0])

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/almeidadm/economy-classifier.git"
    REPO_BRANCH = "main"
    DRIVE_FOLDER = "economy-classifier"

    DRIVE_BASE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    REPO_DIR = Path("/content/economy-classifier")

    if REPO_DIR.exists():
        _run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], "git fetch")
        _run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], "git checkout")
        _run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], "git reset")
    else:
        _run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], "git clone")

    _run(
        [sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR),
         "--upgrade-strategy", "only-if-needed", "-q"],
        "pip install -e .",
    )
    # transformers >= 4.45 needed for Qwen2.5/Llama-3.1 chat templates
    _run(
        [sys.executable, "-m", "pip", "install",
         "transformers>=4.45", "accelerate>=0.34", "-q"],
        "pip install transformers/accelerate",
    )

    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))

    SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
    if not (SPLITS_DIR / "test.parquet").exists():
        zip_path = DRIVE_BASE / "colab_splits.zip"
        assert zip_path.exists(), f"Falta colab_splits.zip em {DRIVE_BASE}."
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(REPO_DIR / "artifacts")

    RUNS_BASE = DRIVE_BASE / "runs"
else:
    REPO_DIR = Path.cwd().parent
    SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
    RUNS_BASE = REPO_DIR / "artifacts" / "runs"

RUNS_BASE.mkdir(parents=True, exist_ok=True)
print("SPLITS_DIR:", SPLITS_DIR)
print("RUNS_BASE :", RUNS_BASE)
print("HARDWARE  :", HARDWARE)


def _sanity_check_imports() -> None:
    failures = []
    for mod_name in ("numpy", "scipy", "pandas", "torch", "transformers", "economy_classifier"):
        try:
            __import__(mod_name)
        except Exception as e:  # noqa: BLE001
            failures.append((mod_name, type(e).__name__, str(e)))
    if failures:
        for name, kind, msg in failures:
            print(f"  - {name}: {kind}: {msg[:200]}")
        raise RuntimeError(
            "Modulos quebrados. Solucao: Runtime > Restart runtime, e re-execute."
        )
    import torch, transformers  # noqa: E401
    print(f"\ntorch={torch.__version__}  transformers={transformers.__version__}")
    print("OK: economy_classifier importavel")


_sanity_check_imports()


## 2. Imports e configuracao

In [ ]:
import gc
import json
import time
from pathlib import Path

import pandas as pd
import torch

from economy_classifier.evaluation import (
    compute_binary_metrics,
    compute_cost_metrics,
    compute_roc_auc,
)
from economy_classifier.llm_review import (
    LLM_REGISTRY,
    classify_batch_hf,
    hf_results_to_predictions,
    load_hf_model,
)
from economy_classifier.project import (
    build_result_card,
    compute_artifact_size_mb,
    persist_result_card,
)

SEED = 42

# ---- Quais LLMs rodar ----
# Comente para pular um modelo. Tempo total no L4 com batch_size=8: ~50min/modelo
# para o test_set completo (~16k registros). Para smoke test, ajuste MAX_SAMPLES.
LLMS = [
    ("qwen2.5-7b-instruct",   LLM_REGISTRY["qwen2.5-7b-instruct"]),
    ("llama-3.1-8b-instruct", LLM_REGISTRY["llama-3.1-8b-instruct"]),
]

# ---- Configuracao da inferencia ----
LOAD_DTYPE       = "float16"  # FP16: 7B ~14 GB, 8B ~16 GB. Use "int8" se VRAM < 20 GB
BATCH_SIZE       = 8           # ajuste se OOM (4 ou 2)
MAX_NEW_TOKENS   = 80          # JSON curto: {"label":"...", "justificativa":"..."}
MAX_INPUT_LENGTH = 2048        # truncamento (Folha media: ~700 tokens)
TEMPERATURE      = 0.0         # deterministic (greedy)

# ---- Sampling ----
MAX_SAMPLES = None  # None = test set completo. Use ex. 1000 para smoke test rapido.

print(f"LLMs a avaliar: {[k for k, _ in LLMS]}")
print(f"BATCH_SIZE={BATCH_SIZE}  LOAD_DTYPE={LOAD_DTYPE}  MAX_SAMPLES={MAX_SAMPLES}")


## 3. Carga do test set

Subsample estratificado opcional via `MAX_SAMPLES`.

In [ ]:
test = pd.read_parquet(SPLITS_DIR / "test.parquet")
print(f"test: {len(test):,} registros (mercado={test['label'].mean()*100:.2f}%)")

if MAX_SAMPLES is not None and MAX_SAMPLES < len(test):
    # Estratificado: preserve a proporcao mercado/outros
    test_eval = (
        test.groupby("label", group_keys=False)
            .apply(lambda g: g.sample(n=int(MAX_SAMPLES * len(g) / len(test)),
                                       random_state=SEED))
            .reset_index(drop=True)
    )
    print(f"Subsample: {len(test_eval):,} registros "
          f"(mercado={test_eval['label'].mean()*100:.2f}%)")
else:
    test_eval = test.copy()
    print(f"Avaliando o test set completo ({len(test_eval):,} registros).")

# Os indices originais (do corpus) ficam em test_eval.index
RECORD_IDS = test_eval.index.tolist()
TEXTS = test_eval["text"].fillna("").tolist()
Y_TRUE = test_eval["label"].values


## 4. Funcao de avaliacao por LLM

Carrega o modelo em VRAM, classifica em batches (com checkpoint), computa metricas, persiste `result_card.json` e libera memoria.

In [ ]:
def evaluate_llm(model_key: str, model_name: str) -> dict:
    """Carrega o modelo, classifica o test_eval, salva predictions + result_card.

    Retorna um sumario com metricas e custo.
    """
    model_id = f"llm_{model_key.replace('.', '_').replace('-', '_')}"
    print(f"\n{'='*72}\nLLM: {model_id}  ({model_name})\n{'='*72}")

    run_dir = RUNS_BASE / f"{model_id}_binary_test_set"
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint = run_dir / "predictions_checkpoint.csv"

    # ---- Carga do modelo ----
    print(f"[{model_id}] Carregando modelo (dtype={LOAD_DTYPE})...")
    t0 = time.perf_counter()
    tokenizer, model = load_hf_model(model_name, dtype=LOAD_DTYPE)
    load_time = time.perf_counter() - t0

    n_params = sum(p.numel() for p in model.parameters())
    model_size_mb = round(n_params * (2 if LOAD_DTYPE == "float16" else 1) / 1e6, 1)
    print(f"[{model_id}] Carregado em {load_time:.1f}s. "
          f"Params={n_params/1e9:.2f}B  Tamanho aproximado={model_size_mb} MB")

    # ---- Inferencia em batches com checkpoint ----
    print(f"[{model_id}] Classificando {len(TEXTS):,} textos "
          f"(batch_size={BATCH_SIZE})...")
    t0 = time.perf_counter()
    raw_results = classify_batch_hf(
        tokenizer, model,
        texts=TEXTS, record_ids=RECORD_IDS,
        method=model_key,
        max_new_tokens=MAX_NEW_TOKENS,
        max_input_length=MAX_INPUT_LENGTH,
        temperature=TEMPERATURE,
        batch_size=BATCH_SIZE,
        checkpoint_path=checkpoint,
        checkpoint_every=200,
    )
    inference_time = time.perf_counter() - t0
    n_errors = sum(1 for r in raw_results if r.get("label") == "erro")
    print(f"[{model_id}] {len(raw_results)} predicoes em {inference_time:.0f}s "
          f"({n_errors} erros de parse)")

    # ---- Converter para o formato padrao de predictions ----
    preds = hf_results_to_predictions(raw_results)
    # Anexar y_true para metricas
    truth = pd.DataFrame({"index": RECORD_IDS, "y_true": Y_TRUE})
    preds = preds.merge(truth, on="index", how="left")
    preds = preds[["index", "y_true", "y_pred", "y_score", "method", "label", "justificativa"]]
    preds.to_csv(run_dir / "predictions.csv", index=False)

    # ---- Metricas ----
    valid = preds.dropna(subset=["y_true"])
    metrics = compute_binary_metrics(valid["y_true"].values, valid["y_pred"].values)
    metrics["auc_roc"] = round(
        compute_roc_auc(valid["y_true"].values, valid["y_score"].values), 4,
    )
    metrics["coverage"] = round(len(valid) / len(test_eval), 4)  # 1 - taxa de erro

    cost = compute_cost_metrics(
        train_seconds=0.0,  # zero-shot — sem treino
        inference_seconds=inference_time,
        n_inference_samples=len(test_eval),
        model_size_mb=model_size_mb,
        n_parameters=int(n_params),
        hardware=HARDWARE,
    )
    cost["model_load_seconds"] = round(load_time, 1)
    cost["batch_size"] = BATCH_SIZE
    cost["max_new_tokens"] = MAX_NEW_TOKENS

    config = {
        "model_name": model_name,
        "dtype": LOAD_DTYPE,
        "batch_size": BATCH_SIZE,
        "max_new_tokens": MAX_NEW_TOKENS,
        "max_input_length": MAX_INPUT_LENGTH,
        "temperature": TEMPERATURE,
        "max_samples": MAX_SAMPLES,
    }

    persist_result_card(build_result_card(
        model_id=model_id, task="binary", regime="test_set",
        metrics=metrics, cost=cost, config=config,
        n_train_samples=0, n_eval_samples=len(test_eval),
        predictions_path=str(run_dir / "predictions.csv"),
        notes=("Zero-shot LLM classification on test set. "
               "y_score is binary (0 or 1) — LLMs do not produce calibrated probabilities natively."),
    ), run_dir)

    print(f"[{model_id}] f1={metrics['f1']:.4f}  precision={metrics['precision']:.4f}  "
          f"recall={metrics['recall']:.4f}  coverage={metrics['coverage']:.4f}")

    # ---- Liberar VRAM ----
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "model_id": model_id,
        "metrics": metrics,
        "cost": cost,
        "n_errors": n_errors,
    }


## 5. Loop sobre os LLMs

**Tempo estimado em L4** com test set completo (~16k registros), batch_size=8:
- Qwen2.5-7B FP16: ~50 min
- Llama-3.1-8B FP16: ~55 min
- Total: ~1h45min

Para smoke test rapido, ajuste `MAX_SAMPLES = 1000` na celula 5.

In [ ]:
all_summaries = []
for model_key, model_name in LLMS:
    summary = evaluate_llm(model_key, model_name)
    all_summaries.append(summary)

print("\n\n=== TODOS OS LLMs AVALIADOS ===")
for s in all_summaries:
    print(json.dumps({
        "model_id": s["model_id"],
        "f1": s["metrics"]["f1"],
        "precision": s["metrics"]["precision"],
        "recall": s["metrics"]["recall"],
        "auc_roc": s["metrics"]["auc_roc"],
        "coverage": s["metrics"]["coverage"],
        "inference_seconds": s["cost"].get("inference_seconds_mean") or s["cost"].get("inference_seconds_total"),
        "n_errors": s["n_errors"],
    }, indent=2))


## 6. Sumario comparativo

In [ ]:
rows = []
for sub in sorted(RUNS_BASE.glob("llm_*_binary_test_set")):
    card_path = sub / "result_card.json"
    if not card_path.exists():
        continue
    c = json.loads(card_path.read_text())
    m = c["metrics"]
    rows.append({
        "model_id": c["model_id"],
        "f1": m.get("f1"),
        "precision": m.get("precision"),
        "recall": m.get("recall"),
        "auc_roc": m.get("auc_roc"),
        "coverage": m.get("coverage"),
        "inf_s": c["cost"].get("inference_seconds_total") or c["cost"].get("inference_seconds_mean"),
        "size_mb": c["cost"].get("model_size_mb"),
        "params_b": round((c["cost"].get("n_parameters") or 0) / 1e9, 2),
        "load_s": c["cost"].get("model_load_seconds"),
    })
summary_df = pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)
summary_df
